In [7]:
import sys
from pathlib import Path
from tqdm import tqdm
from dotenv import load_dotenv
load_dotenv()
root = Path().resolve().parent  # adjust level as needed
sys.path.insert(0, str(root))

In [8]:
import json
with open(root / "datasets" / "hotpotqa_eval.json", "r") as f:
    eval_dataset = json.load(f)

In [3]:
import pickle
with open(root / "datasets" / "hotpotqa.pkl", "rb") as f:
    all_documents = pickle.load(f) 

In [4]:
from rag_basic.rag_dataset import rag_bot_batch

In [5]:
def generate_outputs(eval_dataset, batch_size: int = 10) -> list[dict]:
    queries = [entry["query"] for entry in eval_dataset]
    outputs = []
    for i in tqdm(range(0, len(queries), batch_size), desc="Generating answers"):
        batch = queries[i:i + batch_size]
        batch_outputs = rag_bot_batch(batch)
        outputs.extend(batch_outputs)

    return outputs

In [6]:
outputs = generate_outputs(eval_dataset)

Generating answers: 100%|██████████| 10/10 [01:47<00:00, 10.71s/it]


In [7]:
results = []
for data, output in zip(eval_dataset, outputs):
    results.append({
        "query": data["query"],
        "answer": data["answer"],
        "outputs_answer": output["answer"],
        "golden_chunk_ids": data["golden_chunk_ids"],
        "outputs_chunk_ids": output["chunk_ids"],
        "context": output["context"]
    })

In [8]:
len(results)

100

In [9]:
save_path = root / "datasets" / "answer_sheet.jsonl"

In [10]:
with open(save_path, 'w', encoding='utf-8') as f:
    for entry in results:
        # ensure_ascii=False is important for Korean text
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f"Saved {len(results)} lines to {save_path}")

Saved 100 lines to /home/jake/RAG-end-to-end/datasets/answer_sheet.jsonl


In [11]:
with open(root / "datasets" / "answer_sheet.jsonl", 'r', encoding='utf-8') as f:
    data = [json.loads(line) for line in f]

In [12]:
from evaluators.llm_evaluator import CorrectnessEvaluator
correctness_evaluator = CorrectnessEvaluator()

In [13]:
batch_size = 10
results = []
for i in tqdm(range(0, len(data), batch_size), desc="Evaluating correctness"):
    batch_data = data[i:i + batch_size]
    batch_queries = [entry["query"] for entry in batch_data]
    batch_outputs = [entry["outputs_answer"] for entry in batch_data]
    batch_answers = [entry["answer"] for entry in batch_data]
    
    batch_results = correctness_evaluator.correctness_batch(batch_queries, batch_outputs, batch_answers)
    results.extend(batch_results)

Evaluating correctness:   0%|          | 0/10 [00:00<?, ?it/s]

Evaluating correctness: 100%|██████████| 10/10 [02:14<00:00, 13.41s/it]


In [14]:
for elem, result in zip(data, results):
    elem['correctness'] = result["correctness"]
    elem['explanation'] = result["explanation"]

In [15]:
save_path = root / "datasets" / "correctness_sheet.jsonl"

In [16]:
with open(save_path, 'w', encoding='utf-8') as f:
    for entry in data:
        # ensure_ascii=False is important for Korean text
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f"Saved {len(results)} lines to {save_path}")

Saved 100 lines to /home/jake/RAG-end-to-end/datasets/correctness_sheet.jsonl


In [9]:
with open(root / "datasets" / "correctness_sheet.jsonl", 'r', encoding='utf-8') as f:
    data = [json.loads(line) for line in f]

In [10]:
for entry in data:
    if entry['correctness'] == 0:
        print("Query: ", entry['query'])
        print("Answer: ", entry['answer'])
        print("Outputs Answer: ", entry['outputs_answer'])
        print("Explanation: ", entry['explanation'])
        print("golden_chunk_ids: ", entry['golden_chunk_ids'])
        print("chunk_ids: ", entry['outputs_chunk_ids'])
        print('-'*100)


Query:  What was the duck character's name in the Disney cartoon with the music that sounded like The Mexican Hat Dance?
Answer:  Donna
Outputs Answer:  The duck was Donald Duck — in the 1937 short "Don Donald" (he woos a Mexican duck named Donna).
Explanation:  Step 1: The ground-truth answer is Donna. Step 2: The student’s answer begins by asserting the duck was Donald Duck, which directly conflicts with the ground truth. Step 3: The student then adds in parentheses that the 1937 short 'Don Donald' features Donald wooing a Mexican duck named Donna — this parenthetical correctly names Donna but does not resolve the initial conflicting claim that the duck was Donald. Step 4: Because the student answer contains a statement that contradicts the ground truth (saying the duck was Donald) it does not meet the accuracy criteria. Therefore the answer is incorrect.
golden_chunk_ids:  ['doc_29_chunk_1', 'doc_29_chunk_5']
chunk_ids:  ['doc_29_chunk_5', 'doc_29_chunk_9', 'doc_29_chunk_2', 'doc_29

In [18]:
def correctness(sheet: list[dict]) -> dict:
    correctness_true = 0
    correctness_false = 0
    for entry in sheet:
        if entry["correctness"] == True:
            correctness_true += 1
        else:
            correctness_false += 1
    correctness = correctness_true / (correctness_true + correctness_false)
    return {"correctness": correctness}   

In [19]:
score = correctness(data)
score


{'correctness': 0.94}